 # Attention Architecture
  Model: **distilbert-base-uncased**
  https://huggingface.co/docs/transformers/model_doc/distilbert

  **Architecture summary:**
  - 6 Transformer encoder layers (half of BERT-base's 12 layers)
  - 12 attention heads per layer
  - Hidden size: 768 dimensions, split evenly across the 12 heads (768 / 12 = 64 dimensions per head)
  - DistilBERT removes the token-type embeddings and pooler layer used in BERT, and is distilled from BERT-base to be smaller and faster while keeping most of its performance


In [8]:
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view, model_view
from IPython.display import display, HTML
import torch

display(HTML("""
<style>
    /* Targets Jupyter/VSCode/Colab output areas to ensure text and background contrast properly */
    .output_subarea, .cell-output, .output_html, .bx-html-wrapper {
        background-color: #ffffff !important;
        color: #000000 !important;
    }
    #bertviz-target, .bertviz-container, svg {
        background: #ffffff !important;
    }
</style>
"""))
# 1. Select the pre-trained transformer model and its tokenizer
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# CRITICAL: output_attentions=True captures the layer matrices
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)

# 2. Match your target phrase structure
PHRASE_TO_ANALYZE = "Life is good!"

# 3. Tokenize input strings into model tensor tensors
inputs = tokenizer(PHRASE_TO_ANALYZE, return_tensors='pt')
outputs = model(**inputs)

# 4. Parse output payload for attention probabilities 
attention = outputs.attentions
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# 5. Display the fully interactive, multi-head graphical platform map
# Hover directly over any word column to watch the lines react inside the notebook output cell
#head_view(attention, tokens)

model_view(attention, tokens)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8813.41it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

 # Attention Architecture
 Model: bert-base-uncased

 Diference with distilbert-base-uncased:
  - DistilBERT is a smaller, faster, and more efficient version of BERT
  - BERT has more parameters and generally achieves higher accuracy on NLP tasks

 **Architecture summary (bert-base-uncased):**
  - 12 Transformer encoder layers (twice as many as DistilBERT's 6)
  - 12 attention heads per layer
  - Hidden size: 768 dimensions, split evenly across the 12 heads (768 / 12 = 64 dimensions per head)
  - Total parameters: ~110M (vs. ~66M for DistilBERT)


In [9]:
from transformers import AutoTokenizer, AutoModel
from bertviz import model_view
import torch
from IPython.display import display, HTML

# 1. Force a white background for the dark theme
display(HTML("""
<style>
    .output_subarea, .cell-output, .output_html, .bx-html-wrapper {
        background-color: #ffffff !important;
        color: #000000 !important;
    }
    #bertviz-target, .bertviz-container, svg {
        background: #ffffff !important;
    }
</style>
"""))

# 2. MODEL NAME (change here to try another model)
# You can use: "bert-base-uncased" (12 layers) or "bert-large-uncased" (24 layers)
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)

# 3. Sample sentence
PHRASE_TO_ANALYZE = "Life is good!"

# 4. Process the input and extract the attention weights
inputs = tokenizer(PHRASE_TO_ANALYZE, return_tensors='pt')
outputs = model(**inputs)
attention = outputs.attentions
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# 5. Render the global grid of layers and heads
model_view(attention, tokens)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8436.85it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

# Q, K, V

The cells below inspect the internal **Query (Q)** and **Key (K)** vectors that BERT computes for a single attention head.

- `layer` selects which of the model's 12 Transformer layers to inspect (index 0-11 for bert-base-uncased).
- `head` selects which of the 12 attention heads within that layer to inspect (index 0-11).
- Each head works on a 64-dimensional slice of the 768-dimensional hidden state (768 / 12 heads = 64 dimensions per head).
- The attention score between two tokens is the dot product of one token's Query vector and the other token's Key vector, computed independently per head.

The example below uses `layer=2` (the 3rd encoder layer) and `head=2` (the 3rd attention head in that layer).


In [10]:
from bertviz.transformers_neuron_view import BertModel, BertTokenizer
from bertviz.neuron_view import show
from IPython.display import display, HTML

# region CSS
display(HTML("""
<style>
    .output_subarea, .cell-output, .output_html, .bx-html-wrapper {
        background-color: #ffffff !important;
        color: #000000 !important;
    }
    #bertviz-target, .bertviz-container, svg, .neuron-view {
        background: #ffffff !important;
    }
    
    /* Styles for the color legend */
    .legend-container {
        font-family: sans-serif;
        background: #fdfdfd;
        border: 1px solid #e0e0e0;
        border-radius: 6px;
        padding: 10px 15px;
        margin-bottom: 15px;
        display: inline-block;
    }
    .legend-title {
        font-size: 13px;
        font-weight: bold;
        margin-bottom: 8px;
        color: #333;
    }
    .legend-bar-wrapper {
        display: flex;
        align-items: center;
        gap: 10px;
    }
    .legend-label {
        font-size: 11px;
        color: #555;
        font-weight: 500;
    }
    .legend-gradient {
        width: 250px;
        height: 15px;
        border-radius: 3px;
        /* Exact replication of the BertViz color map (Orange -> White -> Blue) */
        background: linear-gradient(to right, #d95f02 0%, #f7f7f7 50%, #1f78b4 100%);
        border: 1px solid #ccc;
    }
</style>

<div class="legend-container">
    <div class="legend-title">Color code Guide (Tensor Values)</div>
    <div class="legend-bar-wrapper">
        <span class="legend-label">Orange (-) Strong Negative</span>
        <div class="legend-gradient"></div>
        <span class="legend-label">Blue (+) Strong Positive</span>
    </div>
</div>
"""))
# endregion

MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME, output_attentions=True)

PHRASE_TO_ANALYZE = "Life is good!"

# 4. Render the neuron view (Q, K, V)
show(model, "bert", tokenizer, PHRASE_TO_ANALYZE, layer=2, head=2, display_mode="light")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Load the base model and enable hidden states output
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)

# 2. Prepare sample text and tokenize
sentence = "The animal didn't cross the street because it was too tired."
inputs = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Find the index position for the target token "it"
it_index = tokens.index("it")

# 3. Pass text through the model to extract internal layer weights
with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract projection matrices from Layer 2
layer_index = 2
head_index = 2
hidden_state = outputs.hidden_states[layer_index][0]  # Remove batch dimension

# Isolate the exact linear projection layers for Q and K from Layer 2
query_layer = model.encoder.layer[layer_index].attention.self.query
key_layer = model.encoder.layer[layer_index].attention.self.key

# 5. Compute global Q and K vectors for the whole sequence
all_queries = query_layer(hidden_state)
all_keys = key_layer(hidden_state)

# 6. Squeeze dimensions to isolate Head 2 (BERT-base has 64 dims per head: 768 / 12 heads)
dim_per_head = 64
start_dim = head_index * dim_per_head
end_dim = start_dim + dim_per_head

# Isolate the 64-dimensional vectors specifically for the token "it"
q_it = all_queries[it_index, start_dim:end_dim].detach().numpy()
k_it = all_keys[it_index, start_dim:end_dim].detach().numpy()

# =========================================================================
# PRINT TENSOR VALUES TO THE TERMINAL
# =========================================================================
print(f"=== REAL DATA FOR TOKEN '{tokens[it_index].upper()}' (Layer {layer_index}, Head {head_index}) ===")
print(f"Each vector contains {dim_per_head} high-precision floating-point numbers.\n")

print(f"📌 QUERY VECTOR (q) - First 10 dimensions (The blue/orange bars in the UI):")
print(q_it[:10]) 
print("... [and 54 more numbers]\n")

print(f"📌 KEY VECTOR (k) - First 10 dimensions:")
print(k_it[:10])
print("... [and 54 more numbers]\n")

# Mathematical element-wise product demonstration
elementwise_product = q_it * k_it
print(f"📌 ELEMENT-WISE PRODUCT (q x k) - First 10 results:")
print(elementwise_product[:10])
print("... [and 54 more numbers]\n")

# Dot Product calculation (Sum of all 64 elements)
dot_product = elementwise_product.sum()
print(f"🧮 TOTAL DOT PRODUCT (Sum of all 64 values): {dot_product:.4f}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5821.72it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== REAL DATA FOR TOKEN 'IT' (Layer 2, Head 2) ===
Each vector contains 64 high-precision floating-point numbers.

📌 QUERY VECTOR (q) - First 10 dimensions (The blue/orange bars in the UI):
[-1.0936118  -0.11024669  0.39505     0.17267099  0.43742     0.8646387
 -0.22168809 -0.32597733  1.4985843   1.0465232 ]
... [and 54 more numbers]

📌 KEY VECTOR (k) - First 10 dimensions:
[ 0.0194633   0.23580611  0.25458136  0.3528552   0.44021893  1.2068597
 -0.17578574  0.05400801  1.0590055   0.4392456 ]
... [and 54 more numbers]

📌 ELEMENT-WISE PRODUCT (q x k) - First 10 results:
[-0.0212853  -0.02599684  0.10057236  0.06092786  0.19256057  1.0434976
  0.0389696  -0.01760538  1.587009    0.45968074]
... [and 54 more numbers]

🧮 TOTAL DOT PRODUCT (Sum of all 64 values): 22.6629
